# Evaluation and Visualization of Segmented Linear Regression

---

# Import Libraries

In [ ]:
import sys
import os
root = os.path.abspath('..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from piecewise_regression import r_squared_calc


from modules import load, plots, analysis, utils

# styles
plt.style.use('seaborn-v0_8-white')

---

## Parameters

In [ ]:
# FILE_NAME: Name of the input CSV file containing raw profile measurements (e.g., depth vs conductivity).
ID_PROFILE_NAME = 'BW9D_YSI_20230823'
# INPUT_DIR: Folder (relative to project root) where input files are located.
INPUT_DIR_JSON = os.path.join(root, 'data', 'results')

INPUT_DIR_PROCESSED = os.path.join(root, 'data', 'rawdy')
# FILE_PATH: Full path to the input file. If you set this explicitly, it takes precedence over INPUT_DIR/FILE_NAME.
FILE_PATH_JSON = os.path.join(INPUT_DIR_JSON, ID_PROFILE_NAME + '_rowdy_bic.json')

FILE_PROCESSED_PATH = os.path.join(INPUT_DIR_PROCESSED, ID_PROFILE_NAME + '_rowdy.csv')

CALCULATE_OPTIMAL_TRIAL = True

TRIAL = '1'

N_BREAKPOINT = 3

# Column names in the input CSV (edit these if your file headers differ)
VP_NAME = 'Vertical Position [m]'           # independent variable (x)
SEC_NAME = 'Corrected sp Cond [uS/cm]'      # dependent variable (y)


---

## Load data

In [ ]:
df_json_bic = pd.read_json(FILE_PATH_JSON)
df_json_bic

In [ ]:
# Prefer FILE_PATH if set; otherwise, compose from INPUT_DIR and FILE_NAME
csv_path = FILE_PROCESSED_PATH if os.path.isabs(FILE_PROCESSED_PATH) or FILE_PROCESSED_PATH else os.path.join(INPUT_DIR_PROCESSED, ID_PROFILE_NAME + '_rowdy_processed.csv')
print(f"Reading data from: {csv_path}")
df_processed = pd.read_csv(csv_path)

# Extract vectors
x_processed = df_processed[VP_NAME].to_numpy()
y_processed = df_processed[SEC_NAME].to_numpy()

df_processed.head()

---

### Calculate optimal `n_breakpoint`

In [ ]:
if CALCULATE_OPTIMAL_TRIAL:
    trial = analysis.select_best_trial(FILE_PATH_JSON)
    trial_select = df_json_bic[trial[0]]
    #N_BREAKPOINT = df.loc['best_n_breakpoint_bic'].mode().iloc[0] # alternative, select 'best_n_breakpoint_rss'
else:
    trial_select = df_json_bic[f'trial_{TRIAL}']

In [ ]:
# Elbow plot

x_values = np.array(list(trial_select['df']['n_breakpoints'].values()))
y_values = np.array(list(trial_select['df']['bic'].values()))
secondary_x = np.array(list(trial_select['df']['n_breakpoints'].values()))
secondary_y = np.array(list(trial_select['df']['rss'].values()))

plots.plot_data(
    x_values=x_values,
    y_values=y_values,
    plot_mode='lines+markers',
    x_axis_label="Number Breakpoints",
    y_axis_label="BIC",
    secondary_x=secondary_x,
    secondary_y=secondary_y,
    use_secondary_axis=True,
    y2_axis_label="RSS",
    trace_names=['BIC', 'RSS'],
    title=f"Elbow Plot: <b>{ID_PROFILE_NAME}<b>",
)

---

### Evaluation

In [ ]:
# Params
params_ms = utils.get_breakpoint_data(trial_select['df'], N_BREAKPOINT)
params_ms

In [ ]:
# Model Select
ms = utils.rebuild_model(x_processed,y_processed,params_ms)
ms

In [ ]:
# Globals
RSS, TSS, R2, R2_ajus = r_squared_calc.get_r_squared(y_processed, 
                                                    ms.predict(x_processed), 
                                                    len(ms.get_params()))


print("RSS: ", RSS)
print("TSS: ", TSS)
print("R2: ", R2)
print("R2_ajus: ", R2_ajus)

In [ ]:
# Per segment
metric_per_segment = analysis.calculate_metrics_per_segment(ms)
metric_per_segment

In [ ]:
# Breakpoints
breakpoints = analysis.extract_breakpoints(ms)
breakpoints

##

---

### Final results

#### General models

In [ ]:
# Visualizamos los datos procesados junto con los modelos obtenidos
df_ms = pd.DataFrame({'n_breakpoints': trial_select['df']['n_breakpoints'], 
                    'estimates': trial_select['df']['estimates']})

plots.interactive_segmented_regression(x=x_processed, 
                                       y=y_processed, 
                                       df=df_ms, 
                                       title=f"{ID_PROFILE_NAME}",
                                       breakpoints=N_BREAKPOINT)

#### Models per segment

In [ ]:
segments = utils.extract_segments(ms)   
segments

In [ ]:
plots.plot_segments(segments, 
                    metric_per_segment, 
                    title=f"Segment-wise linear fitting: {ID_PROFILE_NAME}")

---

## Other analysis

### 1. Density of points in processed data


In [ ]:
width = 1 # meters

density = analysis.calculate_density(x_processed, y_processed, bin_width=width)

In [ ]:
# Plot data density
plots.plot_histogram(density,
            value_column='x_bin', 
            weight_column='frequency', 
            num_bins=len(density['x_bin']),
            title=f'Data density histogram (bin width = {width} m) | <b>{ID_PROFILE_NAME}<b>',
            x_axis_title='Vertical Position [m]',
            bar_color='lightgreen'
            ) 